In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print(groq_api_key)  

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


In [9]:
from langchain_groq import ChatGroq

model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002530F0CB410>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002530F0D4610>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
from langchain_core.messages import HumanMessage, SystemMessage


from_language = "English"
to_language = "Arabic"

text = """
Technology has changed the way people live, work, and communicate.
"""

messages = [
    SystemMessage(
        content=f"""
        You are a helpful assistant.

        Your tasks:
        1. Translate the user's text from {from_language} to {to_language}.
        2. Summarize the translated text in 3 bullet points.
        """
    ),
    HumanMessage(content=text)
]

response = model.invoke(messages)

print(response.content)

The translation of the given text from English to Arabic is:

تغيرت التكنولوجيا في كيفية عيش الناس وعملهم وتواصلهم.

Here's a summary of the translated text in 3 bullet points:
* تغيرت التكنولوجيا طريقة حياة الناس
* التكنولوجيا غيرت الطريقة التي يعمل بها الناس
* التكنولوجيا أثرت على كيفية تواصل الأفراد مع بعضهم البعض


In [17]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(response)

"The translation of the given text from English to Arabic is:\n\nتغيرت التكنولوجيا في كيفية عيش الناس وعملهم وتواصلهم.\n\nHere's a summary of the translated text in 3 bullet points:\n* تغيرت التكنولوجيا طريقة حياة الناس\n* التكنولوجيا غيرت الطريقة التي يعمل بها الناس\n* التكنولوجيا أثرت على كيفية تواصل الأفراد مع بعضهم البعض"

In [18]:
chain=model|parser
chain.invoke(messages)

'Translation: \nتغيرت التكنولوجيا في كيفية حياة الناس، وعملهم، وتواصلهم.\n\n\nSummary in 3 bullet points:\n* تؤثر التكنولوجيا على نمط حياة الناس وتغيرت الطرق التي يعيشون بها.\n* التكنولوجيا غيرت الطريقة التي يعمل الناس بها، مما جعل العمليات أكثر كفاءة وفعاليّة.\n* التكنولوجيا غيرت أيضًا الطريقة التي يتواصل بها الناس، مما جعل التواصل أسرع وأكثر انتشارًا.'

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a professional translator and summarizer.

Your tasks:

1. Translate the user's text from {from_language} to {to_language}.

2. Create a SHORT summary of the translated text.

Rules:
- The summary MUST contain exactly 3 bullet points.
- Each bullet point must be ONE sentence only.
- Each bullet point must contain at most 15 words.
- Keep only the most important ideas.
- Do NOT repeat details or examples.
- Respond in the target language.

Format:

### Translation
<translated text>

### Summary
• ...
• ...
• ...
"""
    ),
    ("user", "{text}")
])

messages = prompt.invoke({
    "from_language": "English",
    "to_language": "Arabic",
    "text": "Technology has changed the way people live, work, and communicate."
})

response = model.invoke(messages)

print(response.content)

Translation: 
تغيرت التكنولوجيا في كيفية生活 الناس وعملهم وتواصلهم.

Summary in 3 bullet points:
* التكنولوجيا غيرت طريقة حياة الناس: Technology changed the way people live.
* التكنولوجيا غيرت طريقة عمل الناس: Technology changed the way people work.
* التكنولوجيا غيرت طريقة تواصل الناس: Technology changed the way people communicate.


In [24]:
result=prompt.invoke({
    "from_language": "English",
    "to_language": "Arabic",
    "text": "Technology has changed the way people live, work, and communicate. Today, many tasks that once required hours can be completed in just a few minutes with the help of computers and artificial intelligence. While these advancements have improved productivity and made information more accessible, they have also raised concerns about privacy, security, and the impact of automation on jobs. As technology continues to evolve, it is important to use it responsibly and ensure that it benefits society as a whole."
})

In [25]:
result.to_messages()

[SystemMessage(content="\n        You are a helpful assistant.\n\n        Your tasks:\n        1. Translate the user's text from English to Arabic.\n        2. Summarize the translated text in 3 bullet points.\n        ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Technology has changed the way people live, work, and communicate. Today, many tasks that once required hours can be completed in just a few minutes with the help of computers and artificial intelligence. While these advancements have improved productivity and made information more accessible, they have also raised concerns about privacy, security, and the impact of automation on jobs. As technology continues to evolve, it is important to use it responsibly and ensure that it benefits society as a whole.', additional_kwargs={}, response_metadata={})]

In [ ]:
chain=prompt|model|parser
chain.invoke({"from_language": "English",
    "to_language": "french",
    "text": """Technology has changed the way people live, work, and communicate. Today, many tasks that once required hours can be completed in just a few minutes with the help of computers and artificial intelligence.
      While these advancements have improved productivity and made information more accessible, they have also raised concerns about privacy, security, and the impact of automation on jobs.
        As technology continues to evolve, it is important to use it responsibly and ensure that it benefits society as a whole."""})

"Here is the translation of the text from English to French:\n\nLa technologie a changé la façon dont les gens vivent, travaillent et communiquent. Aujourd'hui, de nombreuses tâches qui nécessitaient autrefois des heures peuvent être accomplies en quelques minutes grâce aux ordinateurs et à l'intelligence artificielle.\nAlors que ces progrès ont amélioré la productivité et rendu l'information plus accessible, ils ont également suscité des inquiétudes quant à la vie privée, à la sécurité et à l'impact de l'automatisation sur les emplois.\nAlors que la technologie continue d'évoluer, il est important de l'utiliser de manière responsable et de veiller à ce qu'elle profite à la société dans son ensemble.\n\nAnd here are 3 bullet points summarizing the translated text:\n\n* La technologie a transformé la façon dont les gens vivent et travaillent, permettant de réaliser des tâches en quelques minutes.\n* Les progrès technologiques ont amélioré la productivité et l'accès à l'information, mais